# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

## Setup

In [1]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion

In [2]:
# define file paths
analysis_subdir_path = "analysis_output"
multirun_filename = "multirun_analyses.json"
# use multirun analyses file to get analysis code paths
multirun_analyses_path = join(analysis_subdir_path, multirun_filename)
with open(multirun_analyses_path, "r") as file:
    multirun_analyses = json.load(file)
num_analyses = multirun_analyses['n']
analysis_code_filenames = [f"llm_analysis_{i}.py" for i in range(num_analyses)]
analysis_code_paths = [join(analysis_subdir_path, filename) for filename in \
    analysis_code_filenames]
# get config details
llm_provider = "openai"
llm_model = "gpt-5-mini"
# create llm assistant
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-11-13 05:09:59.10][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/projects/binyu/hao_huang/stat-genie/config/llm_eval_config.yml'.


## Extract Features **X** Used in Model

In [3]:
# create dict to store features
# features = {}

In [4]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # create internal dict for analysis features
#     features[i] = {}
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     ind_vars = multirun_analyses['analyses'][str(i)]['cvars']['ivs']
#     control_vars = multirun_analyses['analyses'][str(i)]['cvars']['controls']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in ind_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     for dict_idx, var in enumerate(ind_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         ind_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
        
#     # save updated independent variables in features dict
#     features[i]['independent_variables'] = ind_vars
    
#     # tkae same approach for control variables
#     for dict_idx, var in enumerate(control_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         control_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
    
#     # save updated control variables in features dict
#     features[i]['control_variables'] = control_vars

In [5]:
# view feature dictionary to ensure correctness
# features

## Extract Response *y* used in Model

In [6]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     response_vars = multirun_analyses['analyses'][str(i)]['cvars']['dv']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in response_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     # for dict_idx, var in enumerate(response_vars):
#     transform_responses = get_feature_transforms(llm_assistant,
#                                                  transform_code,
#                                                  response_vars['columns'],
#                                                  response_vars['description'])
#     response_vars['transform_code'] = [response.text[0].content \
#         for response in transform_responses]

#     # save updated response variables in features dict
#     features[i]['response_variables'] = response_vars

In [7]:
# view feature dictionary to ensure correctness
# features

## Extract Features **X** and *y* Used in Model

In [8]:
features = format_features(multirun_analyses, num_analyses, llm_assistant)

[2025-11-13 05:10:00.08][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 05:10:10.11][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.02 seconds
[2025-11-13 05:10:10.11][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-13 05:10:10.14][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 05:10:16.95][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.81 seconds
[2025-11-13 05:10:16.95][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-13 05:10:16.99][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 05:10:23.30][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.31

In [9]:
features

{0: {'independent_variables': [{'description': 'Name femininity score (z-scored). Continuous index where higher values indicate a more feminine-sounding hurricane name. Primary independent variable testing whether more feminine names are associated with different fatality outcomes.',
    'columns': ['masfem_z'],
    'transform_code': ["# Create z-scored masfem variable to aid interpretation and reduce scale issues\nmasfem_mean = df['masfem'].mean()\nmasfem_std = df['masfem'].std(ddof=0)\nif masfem_std == 0 or np.isnan(masfem_std):\n    # fallback if no variance\n    df['masfem_z'] = df['masfem'] - masfem_mean\nelse:\n    df['masfem_z'] = (df['masfem'] - masfem_mean) / masfem_std"]}],
  'control_variables': [{'description': 'Binary indicator for whether the hurricane name is classified as female (1) or male (0). Included as a control to separate any discrete-name-gender effects from the continuous masfem measure.',
    'is_moderator': False,
    'moderator_on': None,
    'columns': ['ge

## Extract Model Class Used

In [10]:
model_info = format_model_info(multirun_analyses, num_analyses, llm_assistant)
model_info

[2025-11-13 05:12:20.64][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-11-13 05:12:35.32][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  14.68 seconds
[2025-11-13 05:12:35.32][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-13 05:12:35.33][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 05:12:48.70][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.37 seconds
[2025-11-13 05:12:48.70][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


{0: '{\n  "model_library": "statsmodels (statsmodels.formula.api as smf and statsmodels.api as sm)",\n  "model_class": "Primary: GLM NegativeBinomial (statsmodels.families.NegativeBinomial). Robustness checks: GLM Poisson and OLS on log(alldeaths+1).",\n  "model_parameters": "Formula: \'alldeaths ~ masfem_z + gender_female + wind + min + elapsedyrs + C(category) + C(source)\'. NegativeBinomial: family=sm.families.NegativeBinomial() (default log link). Poisson: family=sm.families.Poisson() with .fit(cov_type=\'HC3\') to get robust SEs. OLS: outcome = log(alldeaths + 1), fit(...).fit(cov_type=\'HC3\'). Data preprocessing: dropna(subset=[...]) on required columns; categorical terms included via C(category) and C(source).",\n  "model_formula_fitting_code": "formula = \'alldeaths ~ masfem_z + gender_female + wind + min + elapsedyrs + C(category) + C(source)\'\\n\\n# Negative Binomial GLM (primary)\\nnb_model = smf.glm(formula=formula, data=mod_df, family=sm.families.NegativeBinomial()).fit(

## Extract Final Answer/Conclusion

Each of the BLADE tasks revolves around a question with the following format:

*What is the effect of [something] on [potential response]?*

It seems that often times the feature to use for the response is not deterministic; the model will have to use some sort of proxy to estimate it. The explanatory features are typically a little bit more clear, but still often require transformations and judgment calls on interpretation and use.

Importantly, this type of question ensures there is a binary answer. While the LLM data scientist does not explicitly spit out a yes/no value, it does write two functions: one which preprocesses the data and another that performs some sort of analysis. Theoretically, we could take the output of the analysis and inspect it to determine whether or not the feature of interest had an effect on the response.

In [19]:
# get path to the dataset
dataset_name = multirun_analyses['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

# load the dataset
data = pd.read_csv(dataset_path)

# create dictionaries to store the imported functions
transform_functions = {}
model_functions = {}

# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    # dynamically import the module
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    # extract transform and model functions
    transform_functions[i] = module.transform
    model_functions[i] = module.model

In [58]:
# Run transform functions on the dataset
transformed_datasets = {}
for i, transform_func in transform_functions.items():
    try:
        transformed_datasets[i] = transform_func(data.copy())  # use copy of dataset
        print(f"[Transform {i}] ✅ Completed successfully.")
    except Exception as e:
        # print(f"[Transform {i}] ❌ Failed with error: {e}")
        print(f"[Transform {i}] ❌")
        transformed_datasets[i] = None

# Run model functions on the transformed datasets
model_results = {}
for i, model_func in model_functions.items():
    try:
        if transformed_datasets[i] is None:
            print(f"[Model {i}] ⚠️ Skipping — transform step failed.")
            continue

        model_results[i] = model_func(transformed_datasets[i].copy())  # use copy
        print(f"[Model {i}] ✅ Completed successfully.")
    except Exception as e:
        print(f"[Model {i}] ❌ Failed with error: {e}")
        model_results[i] = None


[Transform 0] ✅ Completed successfully.
[Transform 1] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 0] ❌ Failed with error: 'GLMResults' object has no attribute 'get_robustcov_results'
[Model 1] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [59]:
# view the first model result as a sanity check
model_results[0]

In [60]:
# create storage object for final answers
final_answer_code = {}

# read task from info.json in the dataset directory
info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)
task = info_json['research_questions']

for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model output from object made in previous cell
    model_output = model_results[i]
    
    # call the helper function
    final_answer_code[i] = write_final_answer_code(llm_assistant, task,
                                                   independent_variable,
                                                   dependent_variable,
                                                   model_code, model_output)

[2025-11-13 07:19:15.40][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 07:20:02.41][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  47.01 seconds
[2025-11-13 07:20:02.41][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [61]:
# run the final answer code
final_answer_code

{0: 'def extract_final_answer(model_output):\n    """\n    Extracts coefficient, SE, p-value, and 95% CI for \'masfem_z\' from the models returned\n    by the modeling function. Also provides a short interpretation about whether the\n    estimated effect supports the hypothesis.\n\n    Input:\n      model_output : dict-like object returned by the modeling function. Expected keys:\n        - \'nb_model\' (may be None)\n        - \'poisson_robust\'\n        - \'ols_log_outcome\'\n        - \'model_dataframe\' (optional, used for sample size if model doesn\'t expose nobs)\n\n    Output:\n      dict with keys:\n        - "object": dict with per-model extracted statistics (coef, se, pvalue, ci_lower, ci_upper, nobs)\n        - "description": short explanation of the numbers and whether they support the hypothesis\n    """\n    def safe_get(result, param):\n        """Safely extract coef, se, pvalue, conf_int, nobs for param from a statsmodels result."""\n        info = {}\n        try:\n   

In [62]:
# loop through final answer code and dynamically execute the functions
final_answer_functions = {}
for i in range(num_analyses):
    # create a namespace dictionary to execute the code in
    namespace = {}
    
    # compile and execute the code
    compiled_code = compile(final_answer_code[i], f"<final_answer_code_{i}>", "exec")
    exec(compiled_code, namespace)
    
    # extract function from namespace
    final_answer_functions[i] = namespace['extract_final_answer']

# run the final answer functions on the model results
final_answers = [final_answer_functions[i](model_results[i]) for i in range(num_analyses)]

In [63]:
conclusions = {}
for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model interpretation code
    try:
        interpretation_code = final_answer_code[i]
    except (KeyError, IndexError, TypeError):
        interpretation_code = None
    
    try: 
        interpretation_output = final_answers[i]
    except (KeyError, IndexError, TypeError):
        interpretation_output = None
    
    # call the helper function
    conclusions[i] = make_conclusion(llm_assistant, task, independent_variable,
                                     dependent_variable, model_code,
                                     interpretation_code, interpretation_output)

[2025-11-13 07:20:08.28][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-11-13 07:20:15.15][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.87 seconds
[2025-11-13 07:20:15.15][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [64]:
conclusions

{0: '{\n  "answer": "Not enough information",\n  "justification": "The model-interpretation step failed: it expected a dict but received something else (object=None), so no coefficient, p-value, or CI for masfem_z were extracted. Without the model estimates or test statistics, we cannot determine whether more feminine names are associated with higher fatalities and thus cannot answer the hypothesis."\n}',
 1: '{\n  "answer": "No",\n  "justification": "The preferred (negative-binomial) model shows masfem_z with a small positive point estimate (coef=0.142) and p=0.704, and an interval spanning negative and positive values. A secondary OLS gives a negative but non‑significant estimate (p=0.207). Neither model shows a statistically significant negative effect, so the analysis does not support the hypothesis."\n}'}

## Applying LLM Judge On All Features

In [65]:
import sys
import os

project_root = "/accounts/projects/binyu/hao_huang/stat-genie"
if project_root not in sys.path:
    sys.path.append(project_root)

from similarity.judge import judge_all #need to design so automtically reads dataset name

In [68]:
# for testing judge_all function since I have edited it
import importlib
import similarity.judge
importlib.reload(similarity.judge)
from similarity.judge import judge_all


In [ ]:
# need to dynamically set vars, models, and q in future

features_vars = [features[0], features[1]]
features_models = [model_info[0], model_info[1]]
features_conclusions = [conclusions[0], conclusions[1]]
q = (
    "Hurricanes with more feminine names are perceived as less threatening and hence lead to fewer precautionary measures by the general public."
)

scores = judge_all(q, features_vars, features_models, features_conclusions)

print("\nFinal Similarity Assessment:")
print(json.dumps(scores, indent=2))


{
  "model_library": "statsmodels (statsmodels.formula.api as smf and statsmodels.api as sm)",
  "model_class": "Primary: GLM NegativeBinomial (statsmodels.families.NegativeBinomial). Robustness checks: GLM Poisson and OLS on log(alldeaths+1).",
  "model_parameters": "Formula: 'alldeaths ~ masfem_z + gender_female + wind + min + elapsedyrs + C(category) + C(source)'. NegativeBinomial: family=sm.families.NegativeBinomial() (default log link). Poisson: family=sm.families.Poisson() with .fit(cov_type='HC3') to get robust SEs. OLS: outcome = log(alldeaths + 1), fit(...).fit(cov_type='HC3'). Data preprocessing: dropna(subset=[...]) on required columns; categorical terms included via C(category) and C(source).",
  "model_formula_fitting_code": "formula = 'alldeaths ~ masfem_z + gender_female + wind + min + elapsedyrs + C(category) + C(source)'\n\n# Negative Binomial GLM (primary)\nnb_model = smf.glm(formula=formula, data=mod_df, family=sm.families.NegativeBinomial()).fit()\n\n# Robust Poisso

1. mention invalids
2. run larger multirun and analyze
3. discuss what's in the judge prompt
4. try a summarization?